# 01 — The Damped Harmonic Oscillator

**A foundational ODE, and the full deep-dive into how PINNs work.**

We model the displacement $u(t)$ of a mass on a spring with friction:

$$\frac{d^2u}{dt^2} + \mu\frac{du}{dt} + ku = 0,
\qquad u(0) = 1,\quad u'(0) = 0$$

with $\mu = 2d$ (damping) and $k = \omega_0^2$ (stiffness). We tackle the **high-frequency
regime** ($\omega_0 = 80$, ~13 oscillations in $t\in[0,1]$) where vanilla PINNs collapse to
$u \equiv 0$, and fix it with a **learnable sinusoidal Ansatz**.

> **Requirements** — run `uv sync --all-packages` at the repo root first, and start Jupyter
> from the project environment (`uv run jupyter lab`) so that the `pinn` library is importable.


## How a PINN Actually Solves a Differential Equation

Before touching code, it is worth being precise about the **cost function** and the
**step-by-step solving process** — this is the bridge between the math and the code.

### The Core PINN Cost Function

In standard ML, the loss is just prediction-vs-data error. For a PINN, the loss is a
**multi-part weighted sum** that enforces both data (if any) and physics simultaneously:

$$
\mathcal{L}_{total} = w_{data}\,\mathcal{L}_{data} + w_{physics}\,\mathcal{L}_{physics} + w_{ic}\,\mathcal{L}_{ic} + w_{bc}\,\mathcal{L}_{bc}
$$

| Term | Role |
|------|------|
| $\mathcal{L}_{data}$ | Only for **inverse problems** — match noisy observations |
| $\mathcal{L}_{physics}$ | Mean squared **PDE residual** at collocation points — the "physics-informed" part |
| $\mathcal{L}_{ic}$ | Match the initial state of the system |
| $\mathcal{L}_{bc}$ | Respect the domain edges (walls, periodicity, ...) |
| $w_i$ | Balance hyperparameters — residuals and conditions can have wildly different scales |

In this repo, this sum is exactly what `PINNTrainer.train()` computes each epoch:
`total = sum(weights[name] * loss_fn(model))`.

### The Five-Step Solving Loop

1. **Forward pass (the "guess")** — sample random **collocation points** in the domain and
   evaluate the network. At epoch 0 the output is pure noise.
2. **Physics check (autodiff)** — use `torch.autograd.grad` to get exact derivatives of the
   network output w.r.t. its *inputs* ($u_t$, $u_x$, $u_{xx}$, ...) and plug them into the
   PDE residual. If the guess were correct, the residual would be **zero everywhere**.
3. **Loss calculation** — square and average every constraint violation: residual, IC error, BC error.
4. **Backpropagation (the "correction")** — `total_loss.backward()` differentiates the
   *physics violation itself* w.r.t. every network weight; Adam updates the weights to
   reduce the violation.
5. **Iterate** — repeat for thousands of epochs. The network morphs until the physics checks out.

> **Key insight:** you are not solving the equation directly — you are training a network to
> **obey** the equation. The PINN never "knows" what a shock or a wave is; it only knows the
> physics residual must go to zero.


## 1. Setup

We import everything from the packaged `pinn` library — the network backbone
(`pinn.core.network.PINN`), the generic multi-loss trainer (`pinn.trainer.trainer.PINNTrainer`),
and plotting helpers. No framework code is redefined in this notebook.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.autograd as autograd
import matplotlib.pyplot as plt

from pinn.core.network import PINN
from pinn.trainer.trainer import PINNTrainer
from pinn.utils.plotting import plot_comparison_1d

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Problem Configuration

All physical parameters and hyperparameters in one place. Note how the physics weight is tiny
($10^{-4}$): the residual scales with $k = \omega_0^2 = 6400$, so its raw magnitude dwarfs the
$O(1)$ initial-condition loss. The weights rebalance the terms — change $\omega_0$ and you will
likely need to retune them.


In [ ]:
# --- Physical parameters ---
D = 2.0                # damping coefficient (delta)
W0 = 80.0              # natural frequency
MU, K = 2 * D, W0**2   # ODE coefficients: u'' + MU*u' + K*u = 0
T_DOMAIN = (0.0, 1.0)

# --- Network / training hyperparameters ---
HIDDEN_LAYERS = 3
HIDDEN_NEURONS = 32
EPOCHS = 15_000
LR = 1e-3
N_COLLOCATION = 100
LOSS_WEIGHTS = {"ic": 0.1, "physics": 1e-4}

print(f"ODE: u'' + {MU}*u' + {K}*u = 0   on t in {T_DOMAIN}")

## 3. The Exact Solution (for validation only)

For the under-damped case ($d < \omega_0$) the closed-form solution is

$$u(t) = e^{-dt}\, 2A \cos(\phi + \omega t), \qquad
\omega = \sqrt{\omega_0^2 - d^2},\quad
\phi = \arctan(-d/\omega),\quad
A = \frac{1}{2\cos\phi}$$

The PINN **never sees this** — it is used purely to measure accuracy at the end.


In [ ]:
def exact_solution(d, w0, t):
    """Analytical solution of the under-damped harmonic oscillator."""
    w = np.sqrt(w0**2 - d**2)
    phi = np.arctan(-d / w)
    A = 1 / (2 * np.cos(phi))
    return np.exp(-d * t) * 2 * A * np.cos(phi + w * t)

DAMPED_FREQ = np.sqrt(W0**2 - D**2)
print(f"True damped frequency: w = {DAMPED_FREQ:.4f}")

## 4. Collocation Points and Loss Terms

- **IC loss** (1 point at $t=0$): $(u(0)-1)^2 + (u'(0)-0)^2$
- **Physics loss** (100 uniform points): $\text{mean}\big[(u'' + \mu u' + k u)^2\big]$

Both first and second derivatives come from `autograd.grad` with `create_graph=True`, so the
graph supports second-order differentiation *and* backprop through the derivatives.


In [ ]:
t_ic = torch.tensor([[0.0]], dtype=torch.float32, device=device, requires_grad=True)
t_physics = (
    torch.linspace(*T_DOMAIN, N_COLLOCATION).view(-1, 1).to(device).requires_grad_(True)
)


def pde_residual(model, t):
    u = model(t)
    u_t = autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_tt = autograd.grad(u_t, t, torch.ones_like(u_t), create_graph=True)[0]
    return u_tt + MU * u_t + K * u


def ic_loss(model):
    u = model(t_ic)
    u_t = autograd.grad(u, t_ic, torch.ones_like(u), create_graph=True)[0]
    return ((u - 1.0) ** 2 + (u_t - 0.0) ** 2).squeeze()


def physics_loss(model):
    return torch.mean(pde_residual(model, t_physics) ** 2)

## 5. The Model: MLP × Learnable Sinusoidal Ansatz

A plain `tanh` MLP suffers from **spectral bias** — it learns low frequencies first and at
$\omega_0 = 80$ typically never resolves the oscillations, collapsing to $u \equiv 0$
(which *also* has zero residual for trivial ICs!). The fix:

$$u(t) = \mathrm{NN}(t)\cdot\sin(a t + b)$$

with $a$ (init $70$) and $b$ (init $1$) as trainable parameters. The MLP only needs to learn the
slowly-varying **envelope** $\approx e^{-dt}$, and the optimiser tunes $a$ toward the true damped
frequency $\omega \approx 79.97$.


In [ ]:
class Ansatz(nn.Module):
    """u(t) = NN(t) * sin(a*t + b) with trainable frequency a and phase b."""

    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.a = nn.Parameter(torch.tensor(70.0))
        self.b = nn.Parameter(torch.tensor(1.0))

    def forward(self, t):
        return self.backbone(t) * torch.sin(self.a * t + self.b)


backbone = PINN(input_dim=1, hidden_layers=HIDDEN_LAYERS, hidden_neurons=HIDDEN_NEURONS)
model = Ansatz(backbone).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params}")

## 6. Training

We disable the trainer's live plot (`plot_every=0`) — interactive matplotlib windows are awkward
inside notebooks — and plot the full loss history afterwards instead.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
trainer = PINNTrainer(model, device=device)

trainer.train(
    n_epochs=EPOCHS,
    optimizer=optimizer,
    loss_functions={"ic": ic_loss, "physics": physics_loss},
    weights=LOSS_WEIGHTS,
    plot_every=0,       # no live plot in notebooks
    debug_every=0,
)

In [ ]:
trainer.plot_loss_history(show_total=True)

## 7. Results: PINN vs. Exact Solution

In [ ]:
t_test = torch.linspace(*T_DOMAIN, 300).view(-1, 1).to(device)

with torch.no_grad():
    u_pinn = model(t_test).cpu().numpy()

t_np = t_test.cpu().numpy()
u_exact = exact_solution(D, W0, t_np)

plot_comparison_1d(
    t_np, u_exact, u_pinn,
    title=f"Damped Harmonic Oscillator (w0={W0}, d={D})",
    xlabel="t", ylabel="u(t)",
    exact_label="Exact", pred_label="PINN",
)

## 8. Quantitative Validation

- **Relative $L_2$ error** against the closed-form solution.
- **Learned frequency $a$** — should converge near $\omega = \sqrt{\omega_0^2 - d^2} \approx 79.97$.


In [ ]:
rel_l2 = np.linalg.norm(u_pinn - u_exact) / np.linalg.norm(u_exact)

print(f"Relative L2 error : {rel_l2:.4e}")
print(f"Learned a (freq)  : {model.a.item():.4f}   (true damped freq: {DAMPED_FREQ:.4f})")
print(f"Learned b (phase) : {model.b.item():.4f}")
print(f"Final total loss  : {trainer.loss_history[-1]['total']:.4e}")
print(f"Epochs run        : {len(trainer.loss_history)}")

## 9. Takeaways

1. **The loss weights are the hard part.** The physics residual scales with $k=\omega_0^2$;
   without the $10^{-4}$ weight the IC terms are invisible to the optimiser.
2. **Spectral bias is real.** Without the Ansatz, this exact setup collapses to $u\equiv 0$.
   Embedding known solution structure (here: "it oscillates") is one of the most effective
   PINN tricks.
3. **The learned $a$ is interpretable** — the network *discovered* the damped frequency from
   the residual alone.

**Next:** `02_burgers_analysis.ipynb` moves from ODE to PDE and adds boundary conditions and
shock formation. The equivalent CLI run is `uv run train-harmonic` (see
`experiments/harmonic_oscillator/`).
